# 🌿 Plantica: ConvNeXt-Base SOTA Plant Disease Training Pipeline
### 🎯 129k Images • All Nested Categories (Rice, Mango, Papaya, Citrus, Vegetables, PlantVillage & Non-Plant Filter)
---
**Key Highlights:**
1. **Smart Recursive Scanner:** Automatically discovers all leaf & crop classes from nested subfolders.
2. **Backbone:** `convnext_base.fb_in22k_ft_in1k_384` (384x384 High Resolution Vision Transformer/ConvNeXt).
3. **Optimization:** Mixed Precision AMP, Cosine Annealing LR Schedule, 2-Phase Fine-Tuning on A100 GPU.
4. **Production Export:** Exports high-speed standalone `.onnx`, `classes.json`, and `.zip` artifacts directly.

In [ ]:
# 1. Install Modern Deep Learning & Export Libraries
!pip install -q timm onnx onnxscript onnxruntime albumentations kaggle

In [ ]:
# 2. Kaggle API Configuration & Dataset Download
import os
import json

# Set Kaggle credentials
kaggle_creds = {"username": "kawsarhossain293", "key": "f5f7431f13b63fa06b3a2ea6fcae3952"}
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("📥 Downloading Full 129k Plantica Dataset from Kaggle...")
!kaggle datasets download -d kawsarhossain293/leaf-deases-detection-dataset -p /content/dataset --unzip
print("✅ Download & Extraction Complete!")

In [ ]:
# 3. Smart Recursive Dataset Scanner (Discovers all Nested Plant & Object Classes)
import glob
from pathlib import Path
from collections import defaultdict

dataset_root = "/content/dataset"
valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.JPG', '.JPEG', '.PNG'}

print("🔍 Recursively scanning dataset directory structure...")

image_records = []  # (image_path, class_name)
class_counts = defaultdict(int)

for root, dirs, files in os.walk(dataset_root):
    img_files = [f for f in files if Path(f).suffix in valid_exts]
    if not img_files:
        continue
    
    # Formulate a clean, standardized class name from parent folders
    rel_path = os.path.relpath(root, dataset_root)
    parts = [p.strip() for p in rel_path.split(os.sep) if p.strip() and p not in {'raw', 'dataset_clean', 'dataset', 'train', 'test', 'val'}]
    
    if not parts:
        continue
    
    # Clean class naming rules
    if '256_ObjectCategories' in parts:
        class_name = f"Non_Plant___{parts[-1]}"
    elif len(parts) >= 2:
        parent_group = parts[-2].replace('_leaf', '').replace('_clean', '').title()
        sub_disease = parts[-1].replace(' ', '_')
        class_name = f"{parent_group}___{sub_disease}"
    else:
        class_name = parts[-1].replace(' ', '_')
    
    for img_name in img_files:
        full_path = os.path.join(root, img_name)
        image_records.append((full_path, class_name))
        class_counts[class_name] += 1

unique_classes = sorted(list(class_counts.keys()))
num_classes = len(unique_classes)
class_to_idx = {cls: idx for idx, cls in enumerate(unique_classes)}

print("="*60)
print(f"📊 DATASET SUMMARY:")
print(f"   • Total Valid Images : {len(image_records):,}")
print(f"   • Total Unique Classes: {num_classes}")
print("="*60)
print("🌱 Sample Discovered Classes:")
for sample_cls in unique_classes[:12]:
    print(f"   - {sample_cls} ({class_counts[sample_cls]} images)")
print(f"   ... and {num_classes - 12} more classes!")

In [ ]:
# 4. PyTorch Dataset & Train/Validation Split
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split

IMAGE_SIZE = 384
BATCH_SIZE = 32  # 32 is optimal for 384x384 ConvNeXt-Base on A100

# Safe Random Split into 85% Train, 15% Validation
train_records, val_records = train_test_split(
    image_records, 
    test_size=0.15, 
    random_state=42, 
    shuffle=True
)

print(f"✅ Split: {len(train_records):,} Training Images | {len(val_records):,} Validation Images")

# Custom High-Speed Dataset Class
class PlanticaDataset(Dataset):
    def __init__(self, records, class_map, transform=None):
        self.records = records
        self.class_map = class_map
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        img_path, class_name = self.records[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))
            
        label = self.class_map[class_name]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Professional ImageNet Augmentation Pipeline (Torchvision)
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = PlanticaDataset(train_records, class_to_idx, transform=train_transforms)
val_dataset = PlanticaDataset(val_records, class_to_idx, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print("🚀 DataLoaders ready for A100 Multi-threaded Processing!")


In [ ]:
# 5. Build SOTA ConvNeXt-Base Model Architecture
import timm
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_BACKBONE = 'convnext_base.fb_in22k_ft_in1k_384'

print(f"🧠 Loading {MODEL_BACKBONE} Pretrained on ImageNet-22k...")
model = timm.create_model(
    MODEL_BACKBONE,
    pretrained=True,
    num_classes=num_classes,
    drop_rate=0.3,
    drop_path_rate=0.2
)
model = model.to(device)

print(f"✅ ConvNeXt Model instantiated with {num_classes} output classes on {device}!")

In [ ]:
# 6. Loss Function, Optimizer & Metrics Helper
from tqdm.auto import tqdm

# Label Smoothing prevents overfitting across 300+ classes
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

def run_epoch(model, loader, optimizer, is_train=True):
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    
    pbar = tqdm(loader, desc='Training' if is_train else 'Validation', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
        if is_train:
            optimizer.zero_grad()
            
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            with torch.set_grad_enabled(is_train):
                outputs = model(images)
                loss = criterion(outputs, labels)
                
        if is_train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
        total_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        
        pbar.set_postfix({
            'loss': f'{total_loss/total:.4f}',
            'acc': f'{correct/total*100:.2f}%'
        })
        
    return total_loss / total, correct / total * 100.0

In [ ]:
# 7. Phase 1: Warmup Classifier Head (3 Epochs)
print("🔥 [PHASE 1] Freezing Backbone & Training Classifier Head (3 Epochs)...")

# Freeze backbone layers
for param in model.parameters():
    param.requires_grad = False
for param in model.get_classifier().parameters():
    param.requires_grad = True

optimizer_p1 = torch.optim.AdamW(model.get_classifier().parameters(), lr=1e-3, weight_decay=1e-4)
best_val_acc = 0.0

for epoch in range(1, 4):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer_p1, is_train=True)
    val_loss, val_acc = run_epoch(model, val_loader, optimizer=None, is_train=False)
    print(f"Phase 1 - Epoch [{epoch}/3] | Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_convnext_plantica.pth')
        print(f"   🏆 New Best Head Accuracy: {best_val_acc:.2f}%")

print("✅ Phase 1 Warmup Complete!")

In [ ]:
# 8. Phase 2: Full Backbone Fine-Tuning with Cosine Annealing (12 Epochs)
print("🚀 [PHASE 2] Unfreezing Full Backbone with Cosine Annealing LR (12 Epochs)...")

# Unfreeze all backbone layers
for param in model.parameters():
    param.requires_grad = True

optimizer_p2 = torch.optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if 'head' not in n], 'lr': 1e-4, 'weight_decay': 1e-2},
    {'params': model.get_classifier().parameters(), 'lr': 5e-4, 'weight_decay': 1e-2}
])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=12, eta_min=1e-6)

for epoch in range(1, 13):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer_p2, is_train=True)
    val_loss, val_acc = run_epoch(model, val_loader, optimizer=None, is_train=False)
    scheduler.step()
    
    current_lr = scheduler.get_last_lr()[0]
    print(f"Phase 2 - Epoch [{epoch:02d}/12] (LR: {current_lr:.6f}) | Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_convnext_plantica.pth')
        print(f"   🏆 New Best Validation Accuracy Saved: {best_val_acc:.2f}%!")

print(f"\n🎉 Training Finished! All-Time Best Validation Accuracy: {best_val_acc:.2f}%")

In [ ]:
# 9. Export to High-Speed Production ONNX Model & Auto-Download
import zipfile
from google.colab import files

print("⚡ Exporting Best ConvNeXt Model to Production ONNX format...")

# Load best saved weights
model.load_state_dict(torch.load('best_convnext_plantica.pth', map_location='cpu'))
model.eval()
model.to('cpu')

dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device='cpu')
onnx_filename = 'plantica_convnext_387.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

# Generate classes metadata mapping
classes_metadata = {
    "num_classes": num_classes,
    "classes": unique_classes,
    "image_size": IMAGE_SIZE,
    "model_architecture": MODEL_BACKBONE
}

with open('classes.json', 'w', encoding='utf-8') as f:
    json.dump(classes_metadata, f, indent=2, ensure_ascii=False)

# Package into zip artifact
zip_filename = 'plantica_convnext_artifacts.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(onnx_filename)
    zf.write('classes.json')
    zf.write('best_convnext_plantica.pth')

file_size_mb = os.path.getsize(zip_filename) / (1024*1024)
print(f"\n🎁 All artifacts zipped into '{zip_filename}' ({file_size_mb:.2f} MB)!")
print("👉 Downloading model artifacts directly to your computer...")

files.download(zip_filename)
